# NB5: Fog of War + State Estimates

**Purpose:** Prove the information barrier works and per-actor state estimates diverge meaningfully.

**Tests:**
1. Per-actor ObservationPackets generated with correct filtering
2. Information barrier: no variable the actor shouldn't see is disclosed
3. Perceptual variables visible only to holding actor
4. Higher intelligence investment → tighter estimates
5. Estimates converge toward canonical with observations, diverge without

**Resolves:** U3 (does per-actor state estimation add value?)

In [ ]:
import sys
sys.path.insert(0, '..')

from wargame.scenario import load_scenario, init_db
from wargame.engine import get_all_variables, apply_delta, advance_turn, run_mechanical_phases
from wargame.fog import (
    get_actor_state_estimates,
    get_observable_variables,
    compute_observation_quality,
    update_state_estimates,
    generate_observation_packet,
    check_information_barrier,
)

SCENARIO_PATH = '../scenarios/us_iran_2026.yaml'
spec = load_scenario(SCENARIO_PATH)
conn = init_db(spec)

actor_ids = [a.id for a in spec.actors]
print(f'Actors: {actor_ids}')
print(f'Variables: {len(spec.state_variables)}')

## Test 1: Observable Variable Filtering

Perceptual variables should only be visible to their holder.

In [ ]:
us_observable = get_observable_variables(conn, 'actor_us', actor_ids)
iran_observable = get_observable_variables(conn, 'actor_iran', actor_ids)

print(f'US can observe ({len(us_observable)} vars):')
for v in sorted(us_observable):
    print(f'  {v}')

print(f'\nIran can observe ({len(iran_observable)} vars):')
for v in sorted(iran_observable):
    print(f'  {v}')

# Key checks:
# sv_perceived_us_coercion is Iran's perception — only Iran should see it
# sv_perceived_iran_threat is US's perception — only US should see it
assert 'sv_perceived_iran_threat' in us_observable, 'US should see its own perception of Iran threat'
assert 'sv_perceived_iran_threat' not in iran_observable, 'Iran should NOT see US perception of Iran threat'
assert 'sv_perceived_us_coercion' in iran_observable, 'Iran should see its own perception of US coercion'
assert 'sv_perceived_us_coercion' not in us_observable, 'US should NOT see Iran perception of US coercion'

# Non-perceptual variables should be visible to both
assert 'sv_military_tension' in us_observable
assert 'sv_military_tension' in iran_observable
assert 'sv_sanctions_intensity' in us_observable
assert 'sv_sanctions_intensity' in iran_observable

print('\n✓ Perceptual variables correctly filtered by actor')

## Test 2: Initial State Estimates Match Canonical

In [ ]:
canonical = get_all_variables(conn)
us_estimates = get_actor_state_estimates(conn, 'actor_us')
iran_estimates = get_actor_state_estimates(conn, 'actor_iran')

# At turn 0, all estimates should match canonical (perfect initial info)
for var_id in canonical:
    if var_id in us_estimates:
        assert abs(us_estimates[var_id] - canonical[var_id]) < 1e-6, f'US estimate for {var_id} should match canonical at start'
    if var_id in iran_estimates:
        assert abs(iran_estimates[var_id] - canonical[var_id]) < 1e-6, f'Iran estimate for {var_id} should match canonical at start'

print('✓ Initial state estimates match canonical values')

## Test 3: State Estimates Diverge After Unobserved Changes

Apply a change to the canonical state WITHOUT updating estimates. Estimates should now differ from canonical.

In [ ]:
# Apply a covert change to sanctions_intensity (as if from a hidden action)
pre_sanctions = canonical['sv_sanctions_intensity']
apply_delta(conn, 'sv_sanctions_intensity', 0.15)
conn.commit()

new_canonical = get_all_variables(conn)
us_est = get_actor_state_estimates(conn, 'actor_us')

print(f'Canonical sv_sanctions_intensity: {new_canonical["sv_sanctions_intensity"]:.3f}')
print(f'US estimate sv_sanctions_intensity: {us_est["sv_sanctions_intensity"]:.3f}')
gap = new_canonical['sv_sanctions_intensity'] - us_est['sv_sanctions_intensity']
print(f'Gap: {gap:+.3f}')

assert abs(gap) > 0.1, f'Gap should be significant after unobserved change, got {gap}'
print('\n✓ State estimates diverge from canonical after unobserved changes')

## Test 4: Observation Quality Depends on Intelligence Investment

In [ ]:
# US has intelligence budget of 2 (from scenario spec)
us_quality = compute_observation_quality(conn, 'actor_us', 1)
# Iran has intelligence budget of 1
iran_quality = compute_observation_quality(conn, 'actor_iran', 1)

print(f'US observation quality (intel budget=2): {us_quality:.3f}')
print(f'Iran observation quality (intel budget=1): {iran_quality:.3f}')

assert us_quality > iran_quality, 'US should have better observation quality (higher intel budget)'
print('\n✓ Higher intelligence investment → better observation quality')

## Test 5: Bayesian Updates Pull Estimates Toward Canonical

Apply observation updates with different quality levels and verify convergence.

In [ ]:
# Start fresh
conn2 = init_db(spec)

# Create a gap: change canonical without updating estimates
apply_delta(conn2, 'sv_sanctions_intensity', 0.2)
apply_delta(conn2, 'sv_military_tension', -0.15)
conn2.commit()

canonical2 = get_all_variables(conn2)

# Run 5 rounds of observation updates at different qualities
print(f'{"Turn":<6} {"Quality":<8} {"Sanctions Est":<15} {"Sanctions Canon":<15} {"Gap":<8}')
print('-' * 55)

for turn in range(1, 6):
    advance_turn(conn2)
    quality = 0.3 + turn * 0.05  # increasing quality over time
    updates = update_state_estimates(conn2, 'actor_us', turn, quality)
    conn2.commit()
    
    est = get_actor_state_estimates(conn2, 'actor_us')
    gap = canonical2['sv_sanctions_intensity'] - est.get('sv_sanctions_intensity', 0)
    print(f'{turn:<6} {quality:<8.2f} {est.get("sv_sanctions_intensity", 0):<15.4f} {canonical2["sv_sanctions_intensity"]:<15.4f} {gap:<+8.4f}')

# After 5 updates, estimate should be closer to canonical
final_est = get_actor_state_estimates(conn2, 'actor_us')
final_gap = abs(canonical2['sv_sanctions_intensity'] - final_est['sv_sanctions_intensity'])
initial_gap = 0.2  # the gap we created
print(f'\nInitial gap: {initial_gap:.3f}')
print(f'Final gap: {final_gap:.3f}')
assert final_gap < initial_gap, 'Estimates should converge toward canonical over time'
print('\n✓ Bayesian updates converge estimates toward canonical')

## Test 6: Information Barrier Verification

Generate observation packets and verify no information leaks.

In [ ]:
# Fresh DB
conn3 = init_db(spec)
advance_turn(conn3)

# Create some divergence
apply_delta(conn3, 'sv_sanctions_intensity', 0.1)
conn3.commit()

# Generate observation packets for both actors
us_packet = generate_observation_packet(
    conn3, 'actor_us', 1,
    narrative_observations=['Intelligence suggests sanctions are having moderate effect.'],
    observation_quality=0.5,
)

iran_packet = generate_observation_packet(
    conn3, 'actor_iran', 1,
    narrative_observations=['Economic pressure from sanctions continues to mount.'],
    observation_quality=0.4,
)

conn3.commit()

# Check information barrier
us_violations = check_information_barrier(conn3, 'actor_us', us_packet, actor_ids)
iran_violations = check_information_barrier(conn3, 'actor_iran', iran_packet, actor_ids)

print(f'US observation packet:')
print(f'  Narratives: {us_packet["observations"]}')
print(f'  Estimate updates: {len(us_packet["state_estimate_updates"])} variables')
print(f'  Violations: {us_violations if us_violations else "none"}')

print(f'\nIran observation packet:')
print(f'  Narratives: {iran_packet["observations"]}')
print(f'  Estimate updates: {len(iran_packet["state_estimate_updates"])} variables')
print(f'  Violations: {iran_violations if iran_violations else "none"}')

# Key check: Iran's packet should NOT contain sv_perceived_iran_threat
assert 'sv_perceived_iran_threat' not in iran_packet['state_estimate_updates'], \
    'LEAK: Iran received update for US perception variable'
# US's packet should NOT contain sv_perceived_us_coercion
assert 'sv_perceived_us_coercion' not in us_packet['state_estimate_updates'], \
    'LEAK: US received update for Iran perception variable'

assert len(us_violations) == 0, f'US information barrier violated: {us_violations}'
assert len(iran_violations) == 0, f'Iran information barrier violated: {iran_violations}'

print('\n✓ Information barrier holds — no leaks detected')

## Test 7: Estimate Divergence Between Actors

After several turns of different observation quality, US and Iran should have different estimates.

In [ ]:
# Fresh DB with some canonical changes
conn4 = init_db(spec)

# Simulate 5 turns of mechanical evolution + observation updates
for turn_num in range(1, 6):
    result = run_mechanical_phases(conn4)
    
    # US gets high-quality observations (intel budget 2)
    us_quality = compute_observation_quality(conn4, 'actor_us', result.turn_number)
    update_state_estimates(conn4, 'actor_us', result.turn_number, us_quality)
    
    # Iran gets lower-quality observations (intel budget 1)
    iran_quality = compute_observation_quality(conn4, 'actor_iran', result.turn_number)
    update_state_estimates(conn4, 'actor_iran', result.turn_number, iran_quality)
    
    conn4.commit()

# Compare final estimates
canonical4 = get_all_variables(conn4)
us_est4 = get_actor_state_estimates(conn4, 'actor_us')
iran_est4 = get_actor_state_estimates(conn4, 'actor_iran')

print(f'{"Variable":<35} {"Canonical":>9} {"US Est":>9} {"IR Est":>9} {"US Gap":>8} {"IR Gap":>8}')
print('-' * 85)

shared_vars = sorted(set(us_est4.keys()) & set(iran_est4.keys()) & set(canonical4.keys()))
for var_id in shared_vars:
    c = canonical4[var_id]
    u = us_est4[var_id]
    i = iran_est4[var_id]
    print(f'{var_id:<35} {c:>9.3f} {u:>9.3f} {i:>9.3f} {u-c:>+8.3f} {i-c:>+8.3f}')

# US should generally have tighter estimates (higher observation quality)
us_total_gap = sum(abs(us_est4[v] - canonical4[v]) for v in shared_vars)
iran_total_gap = sum(abs(iran_est4[v] - canonical4[v]) for v in shared_vars)
print(f'\nTotal estimation error — US: {us_total_gap:.3f}, Iran: {iran_total_gap:.3f}')
assert us_total_gap <= iran_total_gap, 'US should have tighter estimates (higher intel budget)'
print('\n✓ US has tighter estimates than Iran (higher intelligence investment)')

## Summary

**Pass criteria:**
- [ ] Perceptual variables correctly filtered by actor
- [ ] Initial estimates match canonical
- [ ] Estimates diverge after unobserved changes
- [ ] Higher intel investment → better observation quality
- [ ] Bayesian updates converge estimates toward canonical
- [ ] Information barrier holds (no leaks)
- [ ] US has tighter estimates than Iran (higher intel budget)

**Resolves:** U3 (per-actor estimation adds value — different intel investments produce different world views)